In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s5e9/sample_submission.csv
/kaggle/input/playground-series-s5e9/train.csv
/kaggle/input/playground-series-s5e9/test.csv


# Input

In [2]:
train=pd.read_csv('/kaggle/input/playground-series-s5e9/train.csv')
test=pd.read_csv('/kaggle/input/playground-series-s5e9/test.csv')
train.shape

(524164, 11)

# Column List

In [3]:
train.columns

Index(['id', 'RhythmScore', 'AudioLoudness', 'VocalContent', 'AcousticQuality',
       'InstrumentalScore', 'LivePerformanceLikelihood', 'MoodScore',
       'TrackDurationMs', 'Energy', 'BeatsPerMinute'],
      dtype='object')

In [4]:

train.isnull().mean()
features=['id', 'RhythmScore', 'AudioLoudness', 'VocalContent', 'AcousticQuality',
       'InstrumentalScore', 'LivePerformanceLikelihood', 'MoodScore',
       'TrackDurationMs', 'Energy']

In [5]:
test.shape

(174722, 10)

# Data spliting

In [6]:

from sklearn.model_selection import train_test_split
x=train[features]
y=train['BeatsPerMinute']


In [7]:
x_train,x_test,y_train,y_test=train_test_split(x, y, test_size=0.33, random_state=42)

In [8]:
# from sklearn.preprocessing import StandardScaler

# scaler = StandardScaler()
# x_train = scaler.fit_transform(x_train)
# # x_test = std.transform(x_test)


In [9]:
x_test.shape

(172975, 10)

# Correlation of features with target

In [10]:

print("features --------- correlation")
for i in features:
    print(f"{i} --------- {train[i].corr(train['BeatsPerMinute'])} ")

features --------- correlation
id --------- -0.00035462261195836784 
RhythmScore --------- 0.005440112910907483 
AudioLoudness --------- -0.003326712683388071 
VocalContent --------- 0.0048763936700683035 
AcousticQuality --------- -0.0008196325866986877 
InstrumentalScore --------- 0.00190023787007353 
LivePerformanceLikelihood --------- 0.0034706801686869033 
MoodScore --------- 0.007058822581937836 
TrackDurationMs --------- 0.006637172217714407 
Energy --------- -0.004375241222586867 


# Model structure

In [11]:
from lightgbm import LGBMRegressor
lgbm = LGBMRegressor(     
    learning_rate=0.008,       
    max_depth=6,                         
    random_state=123,
    n_jobs=-1          
)





# fitting and train prediction for testing

In [12]:
lgbm.fit(x_train,y_train)
pred=lgbm.predict(x_test)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018712 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 351189, number of used features: 10
[LightGBM] [Info] Start training from score 119.076106


# Rmse from training data 

In [13]:
from sklearn.metrics import mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(y_test, pred))
print("RMSE:", rmse)


RMSE: 26.469508794197196


# Submission

In [14]:
submission = pd.DataFrame({'id': test['id'], 'BeatsPerMinute': lgbm.predict(test[features])})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print("Done!")

Done!
